# Device Network Traces

The network traces are obtained from Mobiperf Dataset and used for simulating network capability heterogeneity in ToxiProxy.

### 1. Download Dataset

In [7]:
WORKDIR = "."
RAW_DIR = f"{WORKDIR}/raw"
SOURCE_DIR = RAW_DIR
OUTPUT_CONFIG_PATH = f"{WORKDIR}/trace.json"

In [8]:
!mkdir -p $WORKDIR $RAW_DIR

In [4]:
# Uncomment the following lines to download the raw data from M-Lab.

# !gsutil -m cp gs://openmobiledata_public $RAW_DIR

### 2. Process Measurement Files (Extract -> Filter -> Parse)

In [10]:
import ast
import json
import zipfile
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

def _find_measurement_entry(zip_file: zipfile.ZipFile):
    return next((name for name in zip_file.namelist() if name.split("/")[-1] == "Measurement"), None)

def _parse_tcp_speed_results(raw_results):
    if isinstance(raw_results, str):
        try:
            raw_results = ast.literal_eval(raw_results)
        except (ValueError, SyntaxError):
            return None
    if not isinstance(raw_results, list) or not raw_results:
        return None

    speeds = []
    for value in raw_results:
        try:
            speeds.append(float(value))
        except (TypeError, ValueError):
            continue

    if not speeds:
        return None
    return sum(speeds) / len(speeds)

def _is_wifi_tcpthroughput(record: dict) -> bool:
    if record.get("success") is not True:
        return False

    parameters = record.get("parameters", {})
    device_properties = record.get("device_properties", {})
    test_type = parameters.get("type", "")
    network_type = str(device_properties.get("network_type", "")).upper()

    return test_type == "tcpthroughput" and network_type == "WIFI"

def _to_epoch_ms(end_time: str):
    if not end_time:
        return None
    try:
        return int(datetime.fromisoformat(end_time.replace("Z", "+00:00")).timestamp() * 1000)
    except ValueError:
        return None

def _parse_measurement_json(measurement_bytes: bytes) -> list:
    try:
        payload = json.loads(measurement_bytes.decode("utf-8"))
    except Exception:
        return []
    return payload if isinstance(payload, list) else []

def _read_records_from_file(file_path: Path) -> list:
    try:
        if file_path.suffix.lower() == ".zip":
            with zipfile.ZipFile(file_path, "r") as zf:
                measurement_entry = _find_measurement_entry(zf)
                if not measurement_entry:
                    return []
                return _parse_measurement_json(zf.read(measurement_entry))

        if file_path.suffix.lower() == ".json":
            with open(file_path, "rb") as jf:
                return _parse_measurement_json(jf.read())
    except Exception as exc:
        tqdm.write(f"Skipping {file_path.name}: {exc}")
    return []

def stream_wifi_tcp_profiles_to_output(
    source_dir=SOURCE_DIR,
    output_file=OUTPUT_CONFIG_PATH,
    max_end_time_delta_ms=100,
    preview_limit=3,
 ):
    source_path = Path(source_dir)
    source_files = sorted(list(source_path.rglob("*.zip")) + list(source_path.rglob("*.json")))

    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    total_written = 0
    next_id = 1
    preview = []

    with open(out_path, "w", encoding="utf-8") as outf:
        outf.write("[\n")
        first_record = True

        for source_file in tqdm(source_files, desc="Processing files"):
            records = _read_records_from_file(source_file)
            pending_down = None

            for record in records:
                if not _is_wifi_tcpthroughput(record):
                    continue

                parameters = record.get("parameters", {})
                values = record.get("values", {})
                avg_kbps = _parse_tcp_speed_results(values.get("tcp_speed_results"))
                if avg_kbps is None:
                    continue

                dir_up = bool(parameters.get("dir_up"))
                end_ms = _to_epoch_ms(parameters.get("end_time"))
                device_info = record.get("device_properties", {}).get("device_info", {}) or {}

                if dir_up is False:
                    pending_down = {
                        "download_kbps": avg_kbps,
                        "end_ms": end_ms,
                        "device_info": device_info,
                    }
                    continue

                if pending_down is None:
                    continue

                if end_ms is not None and pending_down["end_ms"] is not None:
                    if abs(end_ms - pending_down["end_ms"]) > max_end_time_delta_ms:
                        pending_down = None
                        continue

                output_record = {
                    "id": next_id,
                    "upload_kbps": float(avg_kbps),
                    "download_kbps": float(pending_down["download_kbps"]),
                    "device_info": pending_down["device_info"],
                    "network_type": "WIFI",
                }

                if not first_record:
                    outf.write(",\n")
                json.dump(output_record, outf, ensure_ascii=False)
                first_record = False

                if len(preview) < preview_limit:
                    preview.append(output_record)

                total_written += 1
                next_id += 1
                pending_down = None

        outf.write("\n]\n")

    return {
        "count": total_written,
        "preview": preview,
        "output_file": str(out_path),
    }

In [ ]:
# Optional cleanup if you previously extracted JSON files manually
# !rm -f $RAW_DIR/*.json

In [11]:
stream_result = stream_wifi_tcp_profiles_to_output(
    source_dir=SOURCE_DIR,
    output_file=OUTPUT_CONFIG_PATH,
    max_end_time_delta_ms=100,
    preview_limit=3,
)
print(f"Generated {stream_result['count']} network profiles to {stream_result['output_file']}")

Processing files: 100%|██████████| 16818/16818 [04:51<00:00, 57.79it/s]

Generated 54288 network profiles to trace.json


In [12]:
stream_result["preview"]

[{'id': 1,
  'upload_kbps': 4558.6828189767375,
  'download_kbps': 7509.610288818472,
  'device_info': {'model': 'TRT-L53',
   'manufacturer': 'HUAWEI',
   'os': 'INCREMENTAL:C69B163, RELEASE:7.0, SDK_INT:24',
   'tac': '86401303'},
  'network_type': 'WIFI'},
 {'id': 2,
  'upload_kbps': 4614.0429625061115,
  'download_kbps': 13357.802685516393,
  'device_info': {'model': 'SM-G532MT',
   'manufacturer': 'samsung',
   'os': 'INCREMENTAL:G532MTVJS1ARF1, RELEASE:6.0.1, SDK_INT:23',
   'tac': '35292909'},
  'network_type': 'WIFI'},
 {'id': 3,
  'upload_kbps': 5066.270870346442,
  'download_kbps': 12214.571835168406,
  'device_info': {'model': '9203A',
   'manufacturer': 'TCL',
   'os': 'INCREMENTAL:vE5L-0, RELEASE:6.0, SDK_INT:23',
   'tac': '01489800'},
  'network_type': 'WIFI'}]

### 3. Stream Output Profiles

In [14]:
# Output schema per record:
#{
#  "id": int,
#  "upload_kbps": float,
#  "download_kbps": float,
#  "device_info": dict,
#  "network_type": "WIFI"
#}

In [13]:
stream_result

{'count': 54288,
 'preview': [{'id': 1,
   'upload_kbps': 4558.6828189767375,
   'download_kbps': 7509.610288818472,
   'device_info': {'model': 'TRT-L53',
    'manufacturer': 'HUAWEI',
    'os': 'INCREMENTAL:C69B163, RELEASE:7.0, SDK_INT:24',
    'tac': '86401303'},
   'network_type': 'WIFI'},
  {'id': 2,
   'upload_kbps': 4614.0429625061115,
   'download_kbps': 13357.802685516393,
   'device_info': {'model': 'SM-G532MT',
    'manufacturer': 'samsung',
    'os': 'INCREMENTAL:G532MTVJS1ARF1, RELEASE:6.0.1, SDK_INT:23',
    'tac': '35292909'},
   'network_type': 'WIFI'},
  {'id': 3,
   'upload_kbps': 5066.270870346442,
   'download_kbps': 12214.571835168406,
   'device_info': {'model': '9203A',
    'manufacturer': 'TCL',
    'os': 'INCREMENTAL:vE5L-0, RELEASE:6.0, SDK_INT:23',
    'tac': '01489800'},
   'network_type': 'WIFI'}],
 'output_file': 'trace.json'}

### 4. Visualization